# Chapter 8 - Deterministic checks, retrieval metrics, and single-example evaluators

Runs fully offline. Covers the cheap checks to run before reaching for an LLM judge (exact match, schema validation, hit@k vs recall@k, why word-overlap and embedding similarity both miss a negation), plus the LangChain single-example evaluators from `langchain_classic.evaluation` (correctness, tone/conciseness, chain-of-thought, structured extraction) and Hugging Face's `pass@k` for code generation. The LLM-judge cells use `OPENAI_API_KEY` when it's set and a labeled, canned stand-in when it isn't, so the notebook runs top to bottom either way.

## Setup

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('../chapter4'))
from meridian import load_corpus, load_eval_set

corpus = load_corpus()
eval_set = load_eval_set()
print(f'{len(corpus)} corpus documents, {len(eval_set)} evaluation questions')

22 corpus documents, 20 evaluation questions


## Deterministic checks: exact match and schema validation

`langchain.evaluation` doesn't exist in `langchain` 1.x. The whole subpackage moved, unchanged and unwarned, to `langchain-classic`.

In [2]:
from langchain_classic.evaluation import (
    ExactMatchStringEvaluator,
    JsonValidityEvaluator,
)

exact = ExactMatchStringEvaluator(ignore_case=True)
print(exact.evaluate_strings(prediction='90 days', reference='90 days'))
print(exact.evaluate_strings(prediction='ninety days', reference='90 days'))

schema_check = JsonValidityEvaluator()
print(schema_check.evaluate_strings(prediction='{"visa_required": false, "days": 90}'))
print(schema_check.evaluate_strings(prediction='{"visa_required": false,}'))

{'score': 1}
{'score': 0}
{'score': 1}
{'score': 0, 'reasoning': 'Expecting property name enclosed in double quotes: line 1 column 25 (char 24)'}


## Retrieval metrics: hit@k and recall@k are different questions

Chapter 4 gave recall@k, precision@k, MRR, and nDCG. Hit@k asks something narrower: did *any* relevant document make the top k at all, a strict 0 or 1 rather than a fraction. A single-vector cosine retriever over the whole corpus (much cruder than chapter 4's own two-level index) makes the difference concrete on the real evaluation set.

In [3]:
import numpy as np
from model2vec import StaticModel

model = StaticModel.from_pretrained('minishlab/potion-retrieval-32M')

def cosine(a, b):
    va, vb = model.encode([a])[0], model.encode([b])[0]
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

def retrieve(question, k=5):
    scored = [(doc['doc_id'], cosine(question, doc['text'][:300])) for doc in corpus]
    scored.sort(key=lambda x: -x[1])
    return [doc_id for doc_id, _ in scored[:k]]

def hit_at_k(ranked_ids, gold_ids, k):
    return 1.0 if set(ranked_ids[:k]) & set(gold_ids) else 0.0

def recall_at_k(ranked_ids, gold_ids, k):
    gold = set(gold_ids)
    return len(set(ranked_ids[:k]) & gold) / len(gold) if gold else 0.0

for k in (1, 5):
    hits = [hit_at_k(retrieve(item['question'], k=k), item['gold_doc_ids'], k) for item in eval_set]
    recalls = [recall_at_k(retrieve(item['question'], k=k), item['gold_doc_ids'], k) for item in eval_set]
    print(f'k={k}:  mean hit@k={np.mean(hits):.2f}   mean recall@k={np.mean(recalls):.2f}')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

k=1:  mean hit@k=0.75   mean recall@k=0.70


k=5:  mean hit@k=1.00   mean recall@k=1.00


## Why word overlap and embedding similarity both miss a negation

Three sentences about the same Meridian visa policy: the reference, a correct paraphrase sharing almost no words with it, and a wrong answer sharing most of the reference's words but flipping the meaning with a single negation.

In [4]:
def word_overlap(a, b):
    wa, wb = set(a.lower().split()), set(b.lower().split())
    return len(wa & wb) / len(wa | wb)

reference = 'UK passport holders can visit Iceland visa-free for up to 90 days.'
paraphrase = "As a UK citizen you don't need a visa for a 90-day trip to Iceland."
negated = 'UK passport holders can visit Iceland only with a visa, valid for 90 days.'

for name, other in [('paraphrase (correct)', paraphrase), ('negated (wrong)', negated)]:
    sim, overlap = cosine(reference, other), word_overlap(reference, other)
    print(f'{name:22s}  cosine={sim:.3f}  word_overlap={overlap:.3f}')

paraphrase (correct)    cosine=0.691  word_overlap=0.136
negated (wrong)         cosine=0.912  word_overlap=0.529


Both metrics score the wrong answer higher than the correct paraphrase. They measure how much two sentences are about the same thing, not whether one of them is true. Checking truth needs a judge model, which is what the rest of this notebook and `advanced_evaluation.ipynb` turn to next.

## LLM-as-judge evaluators from `langchain_classic.evaluation`

The next few cells use `OPENAI_API_KEY` when it's set. Without one, they fall back to `FakeListChatModel`, a LangChain test double that returns a fixed, labeled response instead of calling a real model - the code path and the *shape* of the result are real, the judgment behind the canned response is not.

In [5]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel


def get_judge(canned_response: str):
    """A real model if a key is set, else a labeled stand-in."""
    if os.environ.get('OPENAI_API_KEY'):
        from langchain.chat_models import init_chat_model
        return init_chat_model('openai:gpt-4o-mini'), True
    print('No OPENAI_API_KEY set: using a canned FakeListChatModel stand-in, not a real judge.')
    return FakeListChatModel(responses=[canned_response]), False

### Evaluating correctness against a reference answer

`labeled_score_string` reads the prompt, the prediction, and a reference answer, then returns a score plus reasoning that tolerates paraphrase while still checking substance - a step up from exact match for anything with more than one correct phrasing.

In [6]:
from langchain_classic.evaluation import load_evaluator

judge, is_real = get_judge('The response covers the key facts accurately. Rating: [[9]]')
scorer = load_evaluator('labeled_score_string', llm=judge)

result = scorer.evaluate_strings(
    input='How much is a checked bag on a Meridian package fare?',
    prediction='Checked bags are charged per the current fee schedule; one cabin bag is free.',
    reference='One cabin bag up to 8 kg is included; checked bags follow the current fee schedule.',
)
print(result)

No OPENAI_API_KEY set: using a canned FakeListChatModel stand-in, not a real judge.


{'reasoning': 'The response covers the key facts accurately. Rating: [[9]]', 'score': 9}


### Evaluating tone and conciseness

Criteria evaluators check a response against a named or custom rubric rather than a reference answer, for qualities that don't have one correct phrasing.

In [7]:
prediction = (
    'A checked bag follows the current fee schedule. '
    "Let us know if you'd like the exact rates for your route!"
)

judge, is_real = get_judge('Y\nThe response is concise and stays on topic.')
conciseness_evaluator = load_evaluator('criteria', criteria='conciseness', llm=judge)
print(conciseness_evaluator.evaluate_strings(prediction=prediction, input='How much is a checked bag?'))

judge, is_real = get_judge('Y\nThe response offers further help in a friendly way.')
custom_friendliness = {'friendliness': 'Is the response written in a friendly, approachable tone?'}
friendliness_evaluator = load_evaluator('criteria', criteria=custom_friendliness, llm=judge)
print(friendliness_evaluator.evaluate_strings(prediction=prediction, input='How much is a checked bag?'))

No OPENAI_API_KEY set: using a canned FakeListChatModel stand-in, not a real judge.
{'reasoning': 'The response is concise and stays on topic.', 'value': 'Y', 'score': 1}
No OPENAI_API_KEY set: using a canned FakeListChatModel stand-in, not a real judge.
{'reasoning': 'The response offers further help in a friendly way.', 'value': 'Y', 'score': 1}


### Evaluating chain-of-thought reasoning

`cot_qa` compares an agent's stated reasoning against an expected explanation, judging whether the thought process is sound, not merely whether the final answer matches.

In [8]:
judge, is_real = get_judge(
    'The reasoning correctly identifies Iceland as Schengen and cites '
    'the 90-day rule.\nGRADE: CORRECT'
)
cot_evaluator = load_evaluator('cot_qa', llm=judge)

agent_reasoning = (
    'Iceland is in the Schengen area, and UK passport holders get 90 visa-free days '
    'in any 180-day period under the Schengen short-stay rule.'
)
expected_reasoning = (
    'Should identify Iceland as Schengen and cite the 90-day-in-180-day short-stay allowance.'
)
result = cot_evaluator.evaluate_strings(
    input='Do I need a visa to visit Iceland on a UK passport, and why?',
    prediction=agent_reasoning,
    reference=expected_reasoning,
)
print(result)

No OPENAI_API_KEY set: using a canned FakeListChatModel stand-in, not a real judge.
{'reasoning': 'The reasoning correctly identifies Iceland as Schengen and cites the 90-day rule.\nGRADE: CORRECT', 'value': 'CORRECT', 'score': 1}


### Evaluating structured extraction

`.bind_functions()` from the 2nd edition is deleted, not deprecated, in current LangChain. `.with_structured_output()` is the current pattern, the same one chapter 4 uses for the `Itinerary` model. `FakeListChatModel` can't stand in here since it doesn't implement structured output, so the offline fallback is a plain canned record, matching chapter 4's own `draft_itinerary` convention.

In [9]:
from pydantic import BaseModel, Field


class InsuranceClaim(BaseModel):
    claimant_name: str = Field(..., description='The name of the claimant')
    claim_id: str = Field(..., description='The unique insurance claim identifier')
    policy_number: str = Field(..., description='The policy number associated with the claim')
    claim_amount: str = Field(..., description="The claimed amount, e.g. '$5000'")
    accident_date: str = Field(..., description='The date of the accident (YYYY-MM-DD)')
    status: str = Field('pending', description='The current status of the claim')

sample_claim_text = (
    'I was involved in a car accident on 2023-08-15. My name is Jane Smith, '
    'Claim ID INS78910, Policy Number POL12345, and the damage is estimated at $3500.'
)

if os.environ.get('OPENAI_API_KEY'):
    from langchain.chat_models import init_chat_model
    extractor = init_chat_model('openai:gpt-4o-mini').with_structured_output(InsuranceClaim)
    result = extractor.invoke(sample_claim_text)
else:
    print('No OPENAI_API_KEY set: returning an offline placeholder record, not a real extraction.')
    result = InsuranceClaim(
        claimant_name='Jane Smith', claim_id='INS78910', policy_number='POL12345',
        claim_amount='$3500', accident_date='2023-08-15',
    )
print(result)

No OPENAI_API_KEY set: returning an offline placeholder record, not a real extraction.
claimant_name='Jane Smith' claim_id='INS78910' policy_number='POL12345' claim_amount='$3500' accident_date='2023-08-15' status='pending'


## Evaluating code generation with pass@k

For a coding agent, correctness is closer to a unit test than a similarity score. `pass@k` reports the fraction of problems solved by at least one of the top k candidates. Running arbitrary generated code carries real risk, which is why `HF_ALLOW_CODE_EVAL` is a deliberate opt-in rather than a default.

In [10]:
os.environ['HF_ALLOW_CODE_EVAL'] = '1'

from evaluate import load

code_eval_metric = load('code_eval')
test_cases = ['assert add(2, 3) == 5']
candidates = [['def add(a, b): return a * b', 'def add(a, b): return a + b']]

pass_at_k, _ = code_eval_metric.compute(references=test_cases, predictions=candidates, k=[1, 2])
print(pass_at_k)

{'pass@1': np.float64(0.5), 'pass@2': np.float64(1.0)}
